In [4]:
from pathlib import Path

LAST_NAME = "Muana"
STUDENT_ID = "TUPM-26-2473"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(digit) for digit in STUDENT_ID if digit.isdigit())
NAME_LENGTH = len(LAST_NAME)

print("Seed:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)




Seed: 3
ID Sum: 24
Name Length: 5


In [ ]:
import csv 
import json
import shutil
import os
from collections import Counter

STUDENT_ID = "TUPM-26-2473"
INPUT_CSV = "weblog.csv"
OUTPUT_FOLDER = "output"
FILTERED_FILE = os.path.join(OUTPUT_FOLDER, "filtered_records.csv")
SUMMARY_FILE = os.path.join(OUTPUT_FOLDER, "summary.json")
BACKUP_FILE = os.path.join(OUTPUT_FOLDER, "summary_backup.json")

# STUDENT SPECIFIC NUMBERS 
seed_digit = int(STUDENT_ID[-1])
ID_SUM = sum(int(digit) for digit in STUDENT_ID if digit.isdigit())
max_records = 1000 + ID_SUM
target_class = (seed_digit % 4) + 2

# Counters
status_counts = Counter()
url_counts = Counter()
total_read = 0
invalid_rows = 0
matched = 0

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

try:
    infile = open(INPUT_CSV, newline="", encoding="utf-8-sig")
except FileNotFoundError:
    print("Could not find", INPUT_CSV)
    raise SystemExit
    print("Could not find", INPUT_FILE)
    raise SystemExit

outfile = open(FILTERED_FILE, "w", newline="", encoding="utf-8")
writer = csv.writer(outfile)
writer.writerow(["IP", "Time", "URL", "Status"])

reader = csv.DictReader(infile)

for row in reader:
    if total_read >= max_records:
        break
    total_read += 1

    ip = row.get("IP")
    time = row.get("Time")
    url = row.get("URL")
    status = row.get("Staus") or row.get("Status") 

    if not ip or not time or not url or not status:
        invalid_rows += 1
        continue

    try:
        status = int(status.strip())
    except ValueError:
        invalid_rows += 1
        continue

    if status // 100 == target_class:
        matched += 1
        writer.writerow([ip.strip(), time.strip(), url.strip(), status])
        status_counts[status] += 1
        url_counts[url.strip()] += 1

infile.close()
outfile.close()

top_urls = url_counts.most_common(5)

summary = {
    "student_id": STUDENT_ID,
    "records_processed": total_read,
    "invalid_records": invalid_rows,
    "target_status_class": str(target_class) + "xx",
    "records_in_target_class": matched,
    "count_per_status_code": dict(status_counts),
    "five_most_requested_urls": top_urls,
}

with open(SUMMARY_FILE, "w") as f:
    json.dump(summary, f, indent=4)

shutil.copy(SUMMARY_FILE, BACKUP_FILE)
backup_ok = os.path.exists(BACKUP_FILE)

print("Records processed      :", total_read)
print("Target status class    :", str(target_class) + "xx")
print("Records in target class:", matched)
print("Count per status code  :", dict(status_counts))
print("Five most requested URLs:")
for url, count in top_urls:
    print("   ", count, "-", url)
print("Filtered CSV file      :", FILTERED_FILE)
print("JSON summary file      :", SUMMARY_FILE)
print("Backup verification    :", "PASSED - " + BACKUP_FILE if backup_ok else "FAILED")

Records processed      : 1024
Target status class    : 5xx
Records in target class: 0
Count per status code  : {}
Five most requested URLs:
Filtered CSV file      : output\filtered_records.csv
JSON summary file      : output\summary.json
Backup verification    : PASSED - output\summary_backup.json


In [1]:
import csv
import json
import shutil
import os

# STUDENT SPECIFIC SETTINGS 
STUDENT_ID = "TUPM-26-2473"            
INPUT_FILE = "user_behavior_dataset.csv" 
OUTPUT_FOLDER = "output"
EXPORT_FILE = os.path.join(OUTPUT_FOLDER, "selected_users.csv")
SUMMARY_FILE = os.path.join(OUTPUT_FOLDER, "summary.json")
BACKUP_FILE = os.path.join(OUTPUT_FOLDER, "selected_users_backup.csv")

# STUDENT SPECIFIC NUMBERS  
seed_digit = int(STUDENT_ID[-1])
target_os = "Android" if seed_digit % 2 == 0 else "iOS"
target_class = (seed_digit % 5) + 1

print("Target OS:", target_os)
print("Target behavior class:", target_class)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

try:
    infile = open(INPUT_FILE, newline="", encoding="utf-8-sig")
except FileNotFoundError:
    print("Could not find", INPUT_FILE)
    raise SystemExit

reader = csv.DictReader(infile)

total_records = 0
os_matches = 0
selected = []          # users in target OS AND target class
invalid_rows = 0

for row in reader:
    total_records += 1

    os_name = (row.get("Operating System") or "").strip()
    behavior = (row.get("User Behavior Class") or "").strip()

    if os_name.lower() != target_os.lower():
        continue
    os_matches += 1

    try:
        behavior = int(behavior)
        battery = float(row["Battery Drain (mAh/day)"])
        data = float(row["Data Usage (MB/day)"])
    except (ValueError, KeyError, TypeError):
        invalid_rows += 1          
        continue

    if behavior == target_class:
        selected.append({
            "User ID": row["User ID"].strip(),
            "Device Model": row["Device Model"].strip(),
            "Operating System": os_name,
            "Battery Drain (mAh/day)": battery,
            "Data Usage (MB/day)": data,
        })

infile.close()

if len(selected) > 0:
    avg_battery = round(sum(u["Battery Drain (mAh/day)"] for u in selected) / len(selected), 2)
    avg_data = round(sum(u["Data Usage (MB/day)"] for u in selected) / len(selected), 2)
else:
    avg_battery = None
    avg_data = None
    print("No matching records found, so no averages can be calculated.")

fields = ["User ID", "Device Model", "Operating System",
          "Battery Drain (mAh/day)", "Data Usage (MB/day)"]

with open(EXPORT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(selected)

with open(EXPORT_FILE, newline="", encoding="utf-8") as f:
    exported_rows = sum(1 for _ in csv.DictReader(f))

count_ok = (exported_rows == len(selected))

summary = {
    "student_id": STUDENT_ID,
    "operating_system": target_os,
    "behavior_class": target_class,
    "total_records_in_dataset": total_records,
    "records_matching_os": os_matches,
    "records_in_target_class": len(selected),
    "invalid_records_skipped": invalid_rows,
    "average_battery_drain_mAh_per_day": avg_battery,
    "average_data_usage_MB_per_day": avg_data,
    "exported_row_count": exported_rows,
    "count_verification": "PASSED" if count_ok else "FAILED",
}

with open(SUMMARY_FILE, "w") as f:
    json.dump(summary, f, indent=4)


shutil.copy(EXPORT_FILE, BACKUP_FILE)
backup_exists = os.path.exists(BACKUP_FILE)
backup_size = os.path.getsize(BACKUP_FILE) if backup_exists else 0

print()
print("Operating system and behavior class:", target_os, "/ class", target_class)
print("Total records in dataset           :", total_records)
print("Records matching operating system  :", os_matches)
print("Records in target behavior class   :", len(selected))
print("Average battery drain (mAh/day)    :", avg_battery)
print("Average data usage (MB/day)        :", avg_data)
print("Count verification                 :", "PASSED" if count_ok else "FAILED",
      "(exported", exported_rows, "rows, selected", len(selected), ")")
print("Exported CSV file                  :", EXPORT_FILE)
print("JSON summary file                  :", SUMMARY_FILE)
print("Backup file and size               :", BACKUP_FILE, "-", backup_size, "bytes")

Target OS: iOS
Target behavior class: 4

Operating system and behavior class: iOS / class 4
Total records in dataset           : 700
Records matching operating system  : 146
Records in target behavior class   : 29
Average battery drain (mAh/day)    : 2123.21
Average data usage (MB/day)        : 1221.45
Count verification                 : PASSED (exported 29 rows, selected 29 )
Exported CSV file                  : output\selected_users.csv
JSON summary file                  : output\summary.json
Backup file and size               : output\selected_users_backup.csv - 1037 bytes


In [1]:
import csv
import json
import shutil
import os
from collections import Counter

# STUDENT SPECIFIC SETTINGS
STUDENT_ID = "TUPM-26-2473"                                  
INPUT_FILE = "Global_Cybersecurity_Threats_2015-2024.csv"
YEAR_COL = "Year"
TYPE_COL = "Attack Type"
LOSS_COL = "Financial Loss (in Million $)"

OUTPUT_FOLDER = "output"
FILTERED_CSV = os.path.join(OUTPUT_FOLDER, "filtered_incidents.csv")
TOP5_JSON = os.path.join(OUTPUT_FOLDER, "top5_incidents.json")
JSON_BACKUP = os.path.join(OUTPUT_FOLDER, "top5_incidents_backup.json")
REPORT_FILE = os.path.join(OUTPUT_FOLDER, "report.txt")

# STUDENT SPECIFIC NUMBERS
seed_digit = int(STUDENT_ID[-1])
target_year = 2015 + seed_digit
print("Target year:", target_year)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


try:
    infile = open(INPUT_FILE, newline="", encoding="utf-8-sig")
except FileNotFoundError:
    print("Could not find", INPUT_FILE)
    raise SystemExit

reader = csv.DictReader(infile)
all_columns = reader.fieldnames

total_records = 0
invalid_rows = 0
selected = []                 # incidents in the target year
type_counts = Counter()

for row in reader:
    total_records += 1

    try:
        year = int(row[YEAR_COL].strip())
    except (ValueError, KeyError, AttributeError):
        invalid_rows += 1
        continue
    if year != target_year:
        continue

    try:
        loss = float(row[LOSS_COL].strip())
        if loss < 0:
            raise ValueError
    except (ValueError, KeyError, AttributeError):
        invalid_rows += 1
        continue

    attack = (row.get(TYPE_COL) or "").strip() or "Unknown"
    row[LOSS_COL] = loss
    selected.append(row)

    type_counts[attack] += 1

infile.close()

if len(selected) > 0:
    total_loss = round(sum(r[LOSS_COL] for r in selected), 2)
    average_loss = round(total_loss / len(selected), 2)
else:
    total_loss = 0
    average_loss = 0
    print("No incidents found for", target_year)

top5 = sorted(selected, key=lambda r: r[LOSS_COL], reverse=True)[:5]

with open(TOP5_JSON, "w") as f:
    json.dump({
        "target_year": target_year,
        "total_incidents": len(selected),
        "total_loss_usd_millions": total_loss,
        "average_loss_usd_millions": average_loss,
        "count_per_attack_type": dict(type_counts),
        "five_costliest_incidents": top5,
        "invalid_records_skipped": invalid_rows,
    }, f, indent=4)

with open(FILTERED_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=all_columns)
    writer.writeheader()
    writer.writerows(selected)

shutil.copy(TOP5_JSON, JSON_BACKUP)
backup_ok = (os.path.exists(JSON_BACKUP) and
             os.path.getsize(JSON_BACKUP) == os.path.getsize(TOP5_JSON))

with open(REPORT_FILE, "w") as f:
    f.write("CYBERSECURITY INCIDENT REPORT\n")
    f.write("=" * 40 + "\n")
    f.write("Target year             : " + str(target_year) + "\n")
    f.write("Total records in dataset: " + str(total_records) + "\n")
    f.write("Incidents in target year: " + str(len(selected)) + "\n")
    f.write("Invalid records skipped : " + str(invalid_rows) + "\n\n")

    f.write("Incidents per attack type:\n")
    if type_counts:
        for name, count in type_counts.most_common():
            f.write("  " + name + ": " + str(count) + "\n")
    else:
        f.write("  (none)\n")

    f.write("\nTotal financial loss   : " + str(total_loss) + " million USD\n")
    f.write("Average financial loss : " + str(average_loss) + " million USD\n\n")

    f.write("Five costliest incidents:\n")
    if top5:
        for rank, r in enumerate(top5, start=1):
            f.write("  " + str(rank) + ". " + str(r.get(TYPE_COL)) + " | " +
                    str(r.get("Country", "")) + " | " + str(r[LOSS_COL]) + " million USD\n")
    else:
        f.write("  (none)\n")

    f.write("\nOutput files:\n")
    for path in (FILTERED_CSV, TOP5_JSON, JSON_BACKUP):
        f.write("  " + path + " (" + str(os.path.getsize(path)) + " bytes)\n")
    f.write("Backup verification: " + ("PASSED" if backup_ok else "FAILED") + "\n")

# ---- print in the order of the Assessment Data table ----
print()
print("Target year               :", target_year)
print("Total records in dataset  :", total_records)
print("Incidents in target year  :", len(selected))
print("Count per attack type     :", dict(type_counts))
print("Total financial loss      :", total_loss, "million USD")
print("Average financial loss    :", average_loss, "million USD")
print("Five costliest incidents  :")
for rank, r in enumerate(top5, start=1):
    print("   ", rank, "-", r.get(TYPE_COL), "-", r[LOSS_COL], "million USD")
print("Filtered CSV file         :", FILTERED_CSV)
print("JSON summary file         :", TOP5_JSON)
print("Backup verification       :", "PASSED" if backup_ok else "FAILED")
print("Report file               :", REPORT_FILE)

Target year: 2018

Target year               : 2018
Total records in dataset  : 3000
Incidents in target year  : 310
Count per attack type     : {'Man-in-the-Middle': 36, 'SQL Injection': 57, 'Phishing': 52, 'DDoS': 60, 'Malware': 56, 'Ransomware': 49}
Total financial loss      : 14720.48 million USD
Average financial loss    : 47.49 million USD
Five costliest incidents  :
    1 - Phishing - 99.78 million USD
    2 - SQL Injection - 99.49 million USD
    3 - Ransomware - 99.41 million USD
    4 - Phishing - 99.14 million USD
    5 - Malware - 99.09 million USD
Filtered CSV file         : output\filtered_incidents.csv
JSON summary file         : output\top5_incidents.json
Backup verification       : PASSED
Report file               : output\report.txt
